# 00 — Setup smoke test

Quick environment check for a fresh runtime/team member: clone +
install, mount Drive, stage the data, and verify that the frozen
artifacts and the staged files agree (the same blocking asserts the
real pipeline uses). No training, no GPU needed.

Thin runner: all logic lives in `sharp_har/`.

## 1. Environment setup

In [1]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/FilippoIsoni/sharp-har.git"
REPO_DIR = Path("/content/sharp-har")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

# After the clone, so the file actually exists on a fresh runtime.
!pip install -q -r {REPO_DIR}/requirements.txt

sys.path.insert(0, str(REPO_DIR))

from sharp_har.utils import set_seed, get_git_hash, read_yaml

set_seed(42)
print("git hash:", get_git_hash(REPO_DIR))

git hash: 1944312753201d998fa7084fbb8704f7cdcb6cc1


## 2. Mount Drive + staging

In [2]:
import zipfile
from google.colab import drive

drive.mount("/content/drive")

paths_cfg = read_yaml(REPO_DIR / "configs" / "paths.yaml")
drive_root = Path(paths_cfg["drive_root"])
stage_dir = Path(paths_cfg["stage_dir"])

if not (stage_dir.exists() and any(stage_dir.rglob("*.txt"))):
    stage_dir.mkdir(parents=True, exist_ok=True)
    for zip_name in paths_cfg["zips"]:
        src = drive_root / zip_name
        dst = Path("/content") / zip_name
        print(f"copying {src} -> {dst}")
        subprocess.run(["cp", str(src), str(dst)], check=True)
        with zipfile.ZipFile(dst) as zf:
            zf.extractall(stage_dir)
print("staged:", stage_dir)

Mounted at /content/drive
copying /content/drive/MyDrive/DATASET_SHARP/doppler_traces.zip -> /content/doppler_traces.zip
copying /content/drive/MyDrive/DATASET_SHARP/doppler_traces_S4_S5.zip -> /content/doppler_traces_S4_S5.zip
staged: /content/data


## 3. Smoke checks

Instantiating the datasets runs every day-1/day-2 blocking assert:
file index vs frozen split, 4 antennas per trace, inventory
n_frame consistency, class coverage.

In [3]:
from sharp_har.data import DopplerDataset, build_file_index

files, meta = build_file_index(stage_dir)
print(f"{len(files)} traces staged, {sum(len(v) for v in files.values())} file-streams")

for set_name in ("train", "val", "test"):
    ds = DopplerDataset(
        REPO_DIR / "splits" / "p2_lab.json", set_name, stage_dir,
        inventory_csv=REPO_DIR / "reports" / "inventory.csv",
        arset_map=REPO_DIR / "reports" / "name_to_arset.json",
    )
    s = ds[0]
    assert s["x"].shape == (1, 340, 100)
    print(f"p2_lab {set_name}: {len(ds)} samples, sample x {tuple(s['x'].shape)} OK")

print("environment smoke test PASSED")

2026-07-16 12:38:06,886 [INFO] sharp_har.data: train: 81 traces, 53400 (window, antenna) samples (win=340, stride=100)
2026-07-16 12:38:06,968 [INFO] sharp_har.data: val: 9 traces, 1396 (window, antenna) samples (win=340, stride=340)


102 traces staged, 408 file-streams
p2_lab train: 53400 samples, sample x (1, 340, 100) OK
p2_lab val: 1396 samples, sample x (1, 340, 100) OK


2026-07-16 12:38:07,045 [INFO] sharp_har.data: test: 11 traces, 1700 (window, antenna) samples (win=340, stride=340)


p2_lab test: 1700 samples, sample x (1, 340, 100) OK
environment smoke test PASSED
